In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\IVAN\Desktop\ML обучение с Gemini\Файлы\train.csv",
    sep=","
    )

test = pd.read_csv(
    r"C:\Users\IVAN\Desktop\ML обучение с Gemini\Файлы\test.csv",
    sep=","
    )

X_train = df.drop(columns=['Transported'])
y_train = df['Transported']

X_test = test.copy()

# X.info()
X_train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines


PassengerId разделить на группу и номер в ней
HomePlanet там только 3 варианта(без nan) поэтому иcпользуем ohe и drop='first' чтобы удалит колонку nan если так можно писать, но перед этим заполнить пропуски модой их там около 1-2 процентов
CryoSleep это просто перевести в .astype(int) с пропусками заполнить медианой
Cabin 100 процентов надо разделить на 3 столбца но как их лучше перевести в цифры пока не придумал
Destination также как с HomePlanet
Age оставить таким же, только пропуски не знаю как заполнять, думаю заполнить медианой по планете отправления чтобы не создать огромный 'горб' на графике в одном из возрастов
RoomService, FoodCourt, ShoppingMall, Spa, VRDeck я бы прологарифмировал эти столбцы чтобы сгладить перекос, пропуски медианой
с Name пока не знаю что сделать оставлю таким же, но по факту имя не влияет на предсказание спасётся ли человек, я бы столбец вообще удалил

In [8]:
X_train[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = X_train['Cabin'].str.split('/', expand = True)
X_test[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = X_test['Cabin'].str.split('/', expand = True)

X_train['Cabin_Num'] = X_train['Cabin_Num'].astype(float)
X_test['Cabin_Num'] = X_test['Cabin_Num'].astype(float)

test_passenger_id = X_test['PassengerId']

X_train.drop(columns=['Cabin', 'Name', 'PassengerId'], inplace=True)
X_test.drop(columns=['Cabin', 'Name', 'PassengerId'], inplace=True)
X_train.head()


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Cabin_Deck,Cabin_Num,Cabin_Side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,B,0.0,P
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,F,0.0,S
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,A,0.0,S
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,A,0.0,S
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,F,1.0,S


In [9]:
dec_columns = ['Age', 'Cabin_Num', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
cat_columns = ['HomePlanet', 'Destination', 'CryoSleep', 'VIP', 'Cabin_Deck', 'Cabin_Side']

for col in dec_columns:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

for col in cat_columns:
    mod = X_train[col].mode()[0]  # <--- Ключевая поправка [0]!
    X_train[col] = X_train[col].fillna(mod)
    X_test[col] = X_test[col].fillna(mod)

In [10]:
from sklearn.preprocessing import MinMaxScaler
# import matplotlib.pyplot as plt
# import seaborn as sns

for col in ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

# sns.scatterplot(data=X_train, x='Age', y='Cabin_Num')

scaler = MinMaxScaler()
X_train['Age'] = scaler.fit_transform(X_train[['Age']])
X_test['Age'] = scaler.transform(X_test[['Age']])

X_train['Cabin_Num'] = scaler.fit_transform(X_train[['Cabin_Num']])
X_test['Cabin_Num'] = scaler.transform(X_test[['Cabin_Num']])

X_train['CryoSleep'] = X_train['CryoSleep'].astype(bool).astype(int)
X_test['CryoSleep'] = X_test['CryoSleep'].astype(bool).astype(int)

X_train['VIP'] = X_train['VIP'].astype(bool).astype(int)
X_test['VIP'] = X_test['VIP'].astype(bool).astype(int)

X_train['Cabin_Side'] = X_train['Cabin_Side'].map({'S': 1, 'P': 0})
X_test['Cabin_Side'] = X_test['Cabin_Side'].map({'S': 1, 'P': 0})

cat_cols_to_encode = ['HomePlanet', 'Destination', 'Cabin_Deck']
X_train = pd.get_dummies(X_train, columns=cat_cols_to_encode, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=cat_cols_to_encode, drop_first=True, dtype=int)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

X_test.head(15) 

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Cabin_Num,Cabin_Side,...,HomePlanet_Mars,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_T
0,1,0.341772,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.001584,1,...,0,0,1,0,0,0,0,0,1,0
1,0,0.240506,0,0.000000,2.302585,0.000000,7.945910,0.000000,0.002112,1,...,0,0,1,0,0,0,0,1,0,0
2,1,0.392405,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0,0,0,1,0,0,0,0,0
3,0,0.481013,0,0.000000,8.802823,0.000000,5.204007,6.373320,0.000528,1,...,0,0,1,0,1,0,0,0,0,0
4,0,0.253165,0,2.397895,0.000000,6.455199,0.000000,0.000000,0.002640,1,...,0,0,1,0,0,0,0,1,0,0
5,0,0.392405,0,0.000000,7.387709,5.575949,4.736198,4.110874,0.003696,0,...,0,0,1,0,0,0,0,1,0,0
6,1,0.265823,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.001056,0,...,0,0,0,1,0,0,0,0,0,0
7,1,0.253165,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0,1,0,0,1,0,0,0,0
8,1,0.291139,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0,0,0,0,1,0,0,0,0
9,0,0.303797,0,0.000000,6.461468,0.000000,0.000000,0.000000,0.003696,1,...,0,0,0,0,0,0,0,1,0,0


In [11]:
from sklearn.linear_model import LogisticRegression

# Создаем "пустой мозг" модели
# max_iter=1000 дает алгоритму до 1000 попыток подобрать идеальные веса
model = LogisticRegression(max_iter=1000)

# Переводим таргет из True/False в 1/0, если не сделали этого раньше
y_train = y_train.astype(int)

# Запускаем подбор весов (Процесс обучения)
model.fit(X_train, y_train)

# y_pred — это массив из 1 и 0, который выдала модель
y_pred = model.predict(X_test)
y_pred = y_pred.astype(bool)
print(len(y_pred))

4277


In [12]:
submission = pd.DataFrame({
    'PassengerId': test_passenger_id,
    'Transported': y_pred
})

submission.head()

submission.to_csv(r'C:\Users\IVAN\Desktop\ML обучение с Gemini\Файлы\submission.csv', index=False)